# 🚀 Bane Agent: Fast Neural DPO Benchmark (Google Colab T4 GPU + Drive)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariveshkoshta-spec/Bane_Agent/blob/main/Bane_Colab_GPU_Benchmark.ipynb)

> **Note:** If any cell throws an error, **DO NOT restart the runtime or re-run Step 1!** Libraries remain installed on this VM for your entire session.

### Step 1: Instant Dependency Check & Lightweight Install (~10s first time, 0.1s after)

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ GPU NOT ACTIVE! In Colab menu: Runtime -> Change runtime type -> select 'T4 GPU' -> Save"
print(f"✅ GPU Active: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB VRAM)")

# Check if already installed to prevent unnecessary re-downloads
try:
    import peft
    import bitsandbytes
    import faiss
    import sentence_transformers
    print("⚡ Libraries already installed in this session! Skipped download in 0.1s.")
except ImportError:
    print("⏳ First-time setup: installing lightweight packages (~15s)...")
    !pip install -q peft bitsandbytes faiss-cpu sentence-transformers rich tabulate
    print("✅ Setup complete!")

### Step 2: Clone Bane Agent & Prepare Database

In [ ]:
import os
if not os.path.exists("/content/Bane_Agent") and not os.path.exists("Bane_Agent"):
    !git clone https://github.com/pariveshkoshta-spec/Bane_Agent.git

if os.path.exists("/content/Bane_Agent"):
    %cd /content/Bane_Agent
elif os.path.exists("Bane_Agent"):
    %cd Bane_Agent

!git pull

# Ensure database exists
if not os.path.exists("enterprise_nexus.sqlite"):
    !python scripts/generate_enterprise_nexus.py

print(f"✅ Enterprise Database Ready: {os.path.exists('enterprise_nexus.sqlite')}")

### Step 3: Mount Google Drive & Detect LoRA Adapters 📂

In [ ]:
from google.colab import drive
import os
import shutil
import glob
import zipfile

# 1. Mount Drive
if not os.path.exists('/content/drive/MyDrive'):
    print("⏳ Requesting Google Drive access... click 'Connect to Google Drive' when prompted:")
    drive.mount('/content/drive')
print("✅ Google Drive mounted!")

# 2. Detect adapters folder or zip file
found_adapter = None

# Priority 1: Common folder paths
candidates = [
    "/content/drive/MyDrive/bane_dpo_lora_adapters",
    "/content/drive/MyDrive/Bane_Agent/results/bane_dpo_lora_adapters",
    "/content/drive/MyDrive/results/bane_dpo_lora_adapters",
    "/content/drive/MyDrive/Colab Notebooks/bane_dpo_lora_adapters",
    "/content/bane_dpo_lora_adapters",
    "results/bane_dpo_lora_adapters"
]
for c in candidates:
    if os.path.exists(c) and os.path.exists(os.path.join(c, "adapter_config.json")):
        found_adapter = c
        break

# Priority 2: Check for zip file in Google Drive
if not found_adapter and os.path.exists("/content/drive/MyDrive"):
    zips = glob.glob("/content/drive/MyDrive/**/*adapter*.zip", recursive=True) + \
           glob.glob("/content/drive/MyDrive/**/*bane*.zip", recursive=True)
    if zips:
        target_zip = zips[0]
        print(f"📦 Found adapter zip: {target_zip}. Extracting...")
        extract_dir = "/content/unzipped_adapters"
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(target_zip, 'r') as zf:
            zf.extractall(extract_dir)
        # Check if extracted root or subfolder has config
        for r, dirs, files in os.walk(extract_dir):
            if "adapter_config.json" in files:
                found_adapter = r
                break

# Priority 3: Deep search in Google Drive
if not found_adapter and os.path.exists("/content/drive/MyDrive"):
    configs = glob.glob("/content/drive/MyDrive/**/adapter_config.json", recursive=True)
    if configs:
        found_adapter = os.path.dirname(configs[0])

# 3. Stage to high-speed local SSD
local_adapter_dir = "/content/local_adapters"
if found_adapter:
    print(f"✅ Found adapters at: {found_adapter}")
    if os.path.abspath(found_adapter) != os.path.abspath(local_adapter_dir):
        print("⚡ Staging to local SSD storage for maximum GPU loading speed...")
        os.makedirs(local_adapter_dir, exist_ok=True)
        for item in os.listdir(found_adapter):
            s = os.path.join(found_adapter, item)
            d = os.path.join(local_adapter_dir, item)
            if os.path.isfile(s):
                shutil.copy2(s, d)
        print(f"✅ Successfully staged adapters to: {local_adapter_dir}")
    adapter_path = local_adapter_dir
else:
    adapter_path = None
    print("\n❌ Could not find 'bane_dpo_lora_adapters' or zip in your Google Drive.")
    print("Here are the items currently in your Google Drive root:")
    for f in os.listdir('/content/drive/MyDrive')[:15]:
        print(f"  • {f}")

### Step 4: Load Base Model + LoRA Weights into 16GB GPU VRAM (~15s)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

assert adapter_path is not None and os.path.exists(os.path.join(adapter_path, "adapter_config.json")), \
    f"❌ Please upload bane_dpo_lora_adapters to Google Drive and re-run Step 3!"

base_model_id = "unsloth/llama-3-8b-instruct-bnb-4bit"
print(f"⏳ Loading 4-bit base model ({base_model_id}) on T4 GPU...")

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("⏳ Attaching fine-tuned DPO LoRA adapters...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print(f"🚀 Model + LoRA weights successfully loaded into GPU VRAM!")

### Step 5: Run Full 30-Question Benchmark at GPU Speed (~1s per query) 🚀

In [ ]:
import sqlite3
import time
import json
from src.db_introspector import DatabaseIntrospector
from src.schema_rag import SchemaRetriever
from src.prompt_builder import format_dpo_prompt
from scripts.evaluate_part_a import test_suite_part_a
from scripts.evaluate_part_b import part_b_questions

# 1. Initialize FAISS schema index
introspector = DatabaseIntrospector("enterprise_nexus.sqlite")
schemas = introspector.extract_schemas()
retriever = SchemaRetriever()
retriever.build_index(schemas)
print(f"✅ FAISS indexed {len(schemas)} enterprise tables into vector space.\n")

# 2. Prepare 30 benchmark queries
all_questions = []
for q in test_suite_part_a:
    all_questions.append({
        "id": q["id"],
        "title": q["title"],
        "question": q["question"],
        "expected_sql": q["expected_sql"],
        "type": "Part A (Technical)"
    })
for q in part_b_questions:
    all_questions.append({
        "id": q["id"],
        "title": q["title"],
        "question": q["question"],
        "expected_sql": q["expected_sql"],
        "type": "Part B (Conversational)"
    })

db_conn = sqlite3.connect("enterprise_nexus.sqlite")
cursor = db_conn.cursor()
results = []

print(f"🔥 Evaluating {len(all_questions)} Queries on NVIDIA T4 GPU...\n" + "="*75)

for item in all_questions:
    q_id = item["id"]
    q_text = item["question"]
    exp_sql = item["expected_sql"].strip()

    # RAG schema retrieval
    context = retriever.retrieve_context(q_text, top_k=3)
    prompt = format_dpo_prompt(q_text, context)

    # Neural inference on GPU
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    latency = time.time() - t0
    raw_sql = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    gen_sql = raw_sql.replace(prompt, "").strip()

    # Clean markdown formatting
    if "```sql" in gen_sql:
        gen_sql = gen_sql.split("```sql")[1].split("```")[0].strip()
    elif "```" in gen_sql:
        gen_sql = gen_sql.split("```")[1].split("```")[0].strip()

    # Execute on enterprise_nexus.sqlite
    exec_status = "SUCCESS"
    err_str = None
    gen_rows = []
    try:
        cursor.execute(gen_sql)
        gen_rows = cursor.fetchall()
    except Exception as e:
        exec_status = "EXEC_ERROR"
        err_str = str(e)

    # Execute expected
    cursor.execute(exp_sql)
    exp_rows = cursor.fetchall()

    icon = "✅" if exec_status == "SUCCESS" else "❌"
    print(f"[{icon}] Q{q_id:02d} [{item['type']}]: {item['title']} ({latency:.2f}s | {len(gen_rows)} rows)")
    if exec_status != "SUCCESS":
        print(f"      Error: {err_str}")
        print(f"      SQL:   {gen_sql[:100]}...")

    results.append({
        "id": q_id,
        "title": item["title"],
        "type": item["type"],
        "question": q_text,
        "agent_sql": gen_sql,
        "expected_sql": exp_sql,
        "status": exec_status,
        "error": err_str,
        "gen_rows": len(gen_rows),
        "exp_rows": len(exp_rows),
        "latency": round(latency, 2)
    })

db_conn.close()

# Save results
with open("colab_gpu_results.json", "w") as f:
    json.dump(results, f, indent=2)

total = len(results)
pass_count = sum(1 for r in results if r["status"] == "SUCCESS")
print("\n" + "="*75)
print(f"🎉 Complete! Pass Rate: {pass_count}/{total} ({pass_count/total*100:.1f}%)")

### Step 6: Render Comprehensive Scorecard & Query Review Table

In [ ]:
from IPython.display import display, Markdown

part_a_succ = sum(1 for r in results if "Part A" in r["type"] and r["status"] == "SUCCESS")
part_b_succ = sum(1 for r in results if "Part B" in r["type"] and r["status"] == "SUCCESS")

md_scorecard = f"""
## 📊 GPU Neural Benchmark Scorecard

| Section | Total Queries | Syntax Execution Pass | Accuracy Rate |
| :--- | :---: | :---: | :---: |
| **Part A: Technical & Analytical** | 20 | **{part_a_succ}/20** | **{part_a_succ/20*100:.1f}%** |
| **Part B: Conversational Slang** | 10 | **{part_b_succ}/10** | **{part_b_succ/10*100:.1f}%** |
| **TOTAL OVERALL** | **30** | **{pass_count}/30** | **{pass_count/30*100:.1f}%** |
"""
display(Markdown(md_scorecard))

# Show sample query outputs
print("\nSample Query Outputs:")
for r in results[:5]:
    display(Markdown(f"### Q{r['id']}: {r['title']}\n**Question:** *{r['question']}*\n\n**Generated SQL:**\n```sql\n{r['agent_sql']}\n```\n**Target SQL:**\n```sql\n{r['expected_sql']}\n```"))